In [1]:
import pandas as pd
import numpy as np
import torch
from torch_geometric.data import Data
import torch.nn.functional as F
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import json
from torch_geometric.nn import SAGEConv
import math
from gensim.models import Word2Vec
from torch_geometric import utils
import pickle
import random
from collections import defaultdict
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import roc_auc_score, roc_curve
warnings.filterwarnings('ignore')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
%matplotlib inline

/home/cs/grad/nasrm1/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/lib/python3/dist-packages/paramiko/pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  "cipher": algorithms.TripleDES,
/usr/lib/python3/dist-packages/paramiko/transport.py:261: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  "class": algorithms.TripleDES,


In [2]:
def prepare_graph(df):
    nodes = {}
    labels = {}
    edges = []
    number_of_excluded__rows = 0
    not_existing_labels = set()
    
    dummies = {'SUBJECT_PROCESS': 0, 'FILE_OBJECT_CHAR': 1, 'VALUE_TYPE_SRC': 2, 'SRCSINK_DATABASE': 3,
               'FILE_OBJECT_UNIX_SOCKET': 4,'FILE_OBJECT_BLOCK': 5, 'NetFlowObject': 6, 
               'SRCSINK_PROCESS_MANAGEMENT': 7, 'SUBJECT_THREAD': 8, 'MemoryObject': 9, 
               'FILE_OBJECT_PEFILE': 10, 'FILE_OBJECT_FILE': 11, 'FILE_OBJECT_NAMED_PIPE': 12}

    for i in range(len(df)):
        x = df.iloc[i]
        if x['object'] in dummies:
            action = x["action"]

            actorid = x["actorID"]
            if not (actorid in nodes):
                nodes[actorid] =  []
            if x['exec'] != '':
                nodes[actorid].append(x['exec'])
            nodes[actorid].append(action)
            if x['path'] != '':
                nodes[actorid].append(x['path'])
            labels[actorid] = dummies[x['actor_type']]

            objectid = x["objectID"]
            if not (objectid in nodes):
                nodes[objectid] =  []
            if x['exec'] != '':
                nodes[objectid].append(x['exec'])
            nodes[objectid].append(action)
            if x['path'] != '':
                 nodes[objectid].append(x['path'])
            labels[objectid] = dummies[x['object']]

            edges.append(( actorid, objectid, action ))
        else:
            not_existing_labels.add(x['object'])
            number_of_excluded__rows += 1

    features = []
    feat_labels = []
    edge_index = [[],[]]
    index  = {}
    mapp = []

    for k,v in nodes.items():
      features.append(v)
      feat_labels.append(labels[k])
      index[k] = len(features) - 1
      mapp.append(k)

    for x in edges:
        src = index[x[0]]
        dst = index[x[1]]
        
        edge_index[0].append(src)
        edge_index[1].append(dst) 

    # print(f"Number of excluded rows: {number_of_excluded__rows}")  
    # print(not_existing_labels)          

    return features,feat_labels,edge_index,mapp


In [3]:
def Get_Adjacent(ids, mapp, edges, hops):
    if hops == 0:
        return set()
    neighbors = set()
    for edge in zip(edges[0], edges[1]):
        if any(mapp[node] in ids for node in edge): 
            neighbors.update(mapp[node] for node in edge)
    if hops > 1:
        neighbors = neighbors.union(Get_Adjacent(neighbors, mapp, edges, hops - 1))
    return neighbors

def calculate_metrics(TP, FP, FN, TN):
    FPR = FP / (FP + TN) if FP + TN > 0 else 0
    TPR = TP / (TP + FN) if TP + FN > 0 else 0
    TNR = TN / (TN + FP) if TN + FP > 0 else 0
    prec = TP / (TP + FP) if TP + FP > 0 else 0
    rec = TP / (TP + FN) if TP + FN > 0 else 0
    fscore = (2 * prec * rec) / (prec + rec) if prec + rec > 0 else 0
    return prec, rec, fscore, FPR, TPR, TNR

def helper(MP, all_pids, GP, edges, mapp):
    TP = MP.intersection(GP)  
    FP = MP - GP              
    FN = GP - MP              
    TN = all_pids - (GP | MP) 
    two_hop_gp = Get_Adjacent(GP, mapp, edges, 1)   
    two_hop_tp = Get_Adjacent(TP, mapp, edges, 1)     
    FPL = FP - two_hop_gp                            
    TPL = TP.union(FN.intersection(two_hop_tp))        
    FN = FN - two_hop_tp                      
    TP, FP, FN, TN = len(TPL), len(FPL), len(FN), len(TN)
    prec, rec, fscore, FPR, TPR, TNR = calculate_metrics(TP, FP, FN, TN)
    
    print(f"True Positives: {TP}, True Negatives: {TN}, False Positives: {FP}, False Negatives: {FN}")
    print(f"Precision: {round(prec, 2)}, Recall: {round(rec, 2)}, Fscore: {round(fscore, 2)}")
    print(f"False Positive Rate: {round(FPR, 2)}, True Positive Rate: {round(TPR, 2)}, True Negative Rate: {round(TNR, 2)}")

    return TP, TN, FP, FN, prec, rec, fscore


In [4]:
class GCN(torch.nn.Module):
    def __init__(self,in_channel,out_channel):
        super().__init__()
        self.conv1 = SAGEConv(in_channel, 32, normalize=True)
        self.conv2 = SAGEConv(32, out_channel, normalize=True)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = x.relu()
        x = F.dropout(x, p=0.5, training=self.training)

        x = self.conv2(x, edge_index)
        return x
    
model = GCN(30,13).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
model.to(device)


GCN(
  (conv1): SAGEConv(30, 32)
  (conv2): SAGEConv(32, 13)
)

In [9]:
class PositionalEncoder:

    def __init__(self, d_model, max_len=100000):
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        self.pe = torch.zeros(max_len, d_model)
        self.pe[:, 0::2] = torch.sin(position * div_term)
        self.pe[:, 1::2] = torch.cos(position * div_term)

    def embed(self, x):
        return x + self.pe[:x.size(0)]


def infer(document):
    word_embeddings = [w2vmodel.wv[word] for word in document if word in  w2vmodel.wv]
    
    if not word_embeddings:
        return np.zeros(20)

    output_embedding = torch.tensor(word_embeddings, dtype=torch.float)
    if len(document) < 100000:
        output_embedding = encoder.embed(output_embedding)

    output_embedding = output_embedding.detach().cpu().numpy()
    return np.mean(output_embedding, axis=0)

encoder = PositionalEncoder(30)
w2vmodel = Word2Vec.load("/home/student/nasrm1/Flash-IDS/ALL FLASH FILES/trained_weights/fivedirections/word2vec_five_E3.model")

# Before Evasion

In [ ]:
import pickle 

with open("/home/student/nasrm1/Flash-IDS/files/for_fivedirections.pkl", "rb") as f:
    df_test, nodes_test, phrases_test, labels_test, edges_test, mapp_test, indices_of_malicious_nodes, all_ids, GT_mal = pickle.load(f)
    


In [ ]:
graph = Data(
    x=torch.tensor(nodes_test, dtype=torch.float).to(device),
    y=torch.tensor(labels_test, dtype=torch.long).to(device),
    edge_index=torch.tensor(edges_test, dtype=torch.long).to(device)
)

graph.n_id = torch.arange(graph.num_nodes, device=device)
flag = torch.ones(graph.num_nodes, dtype=torch.bool, device=device)

for m_n in range(30,50):
    model.load_state_dict(torch.load(f'path to trained weights/lword2vec_gnn_five{m_n}_E3.pth',map_location=device))
    model.eval()
    out = model(graph.x, graph.edge_index)  # Directly use the full graph
    sorted, indices = out.sort(dim=1, descending=True)

    conf = (sorted[:, 0] - sorted[:, 1]) / sorted[:, 0]
    conf = (conf - conf.min()) / conf.max()  # normalize

    pred = indices[:, 0]
    cond = (pred == graph.y) & (conf > 0.8) 
    flag[cond] = False

    correct = pred == graph.y  # boolean mask


indices_of_flagged_nodes = torch.nonzero(flag, as_tuple=False).view(-1).tolist()

print("number of nodes flagged as malicious: ", len(indices_of_flagged_nodes))

ids = set([mapp_test[x] for x in indices_of_flagged_nodes]) 
TP, TN, FP, FN, prec, rec, fscore = helper(set(ids), set(all_ids), GT_mal, edges_test, mapp_test)


number of nodes flagged as malicious:  295444
True Positives: 267, True Negatives: 2815, False Positives: 287205, False Negatives: 63
Precision: 0.0, Recall: 0.81, Fscore: 0.0
False Positive Rate: 0.99, True Positive Rate: 0.81, True Negative Rate: 0.01


# After Evasion

In [ ]:
contorter_df = pd.read_pickle('/home/student/nasrm1/Flash-IDS/contorter_fivedirections_flash.pkl')

phrases_mimicry, labels_mimicry, edges_mimicry, mapp_mimicry = prepare_graph(contorter_df)

mimicry_nodes_first_hop = [infer(x) for x in phrases_mimicry]
mimicry_nodes_first_hop = np.array(mimicry_nodes_first_hop)

In [ ]:
graph = Data(
    x=torch.tensor(mimicry_nodes_first_hop, dtype=torch.float).to(device),
    y=torch.tensor(labels_mimicry, dtype=torch.long).to(device),
    edge_index=torch.tensor(edges_mimicry, dtype=torch.long).to(device)
)

graph.n_id = torch.arange(graph.num_nodes, device=device)

flag = torch.ones(graph.num_nodes, dtype=torch.bool, device=device)

for m_n in range(30,50):
    model.load_state_dict(torch.load(f'path to trained weights/lword2vec_gnn_five{m_n}_E3.pth',map_location=device))
    model.eval()
    out = model(graph.x, graph.edge_index)  # Directly use the full graph
    sorted, indices = out.sort(dim=1, descending=True)

    conf = (sorted[:, 0] - sorted[:, 1]) / sorted[:, 0]
    conf = (conf - conf.min()) / conf.max()  # normalize

    pred = indices[:, 0]
    cond = (pred == graph.y) & (conf > 0.8)
    flag[cond] = False

    correct = pred == graph.y  # boolean mask

indices_of_flagged_nodes = torch.nonzero(flag, as_tuple=False).view(-1).tolist()

print("number of nodes flagged as malicious: ", len(indices_of_flagged_nodes))

ids = set([mapp_mimicry[x] for x in indices_of_flagged_nodes]) 
TP, TN, FP, FN, prec, rec, fscore = helper(set(ids), set(all_ids), GT_mal, edges_mimicry, mapp_mimicry)


# at 1 to 3 FOpt
# number of nodes flagged as malicious:  295225
# True Positives: 69, True Negatives: 2825, False Positives: 287196, False Negatives: 261
# Precision: 0.0, Recall: 0.21, Fscore: 0.0
# False Positive Rate: 0.99, True Positive Rate: 0.21, True Negative Rate: 0.01
# Results written to prediction_confidences_curated.txt

number of nodes flagged as malicious:  295225
True Positives: 69, True Negatives: 2825, False Positives: 287196, False Negatives: 261
Precision: 0.0, Recall: 0.21, Fscore: 0.0
False Positive Rate: 0.99, True Positive Rate: 0.21, True Negative Rate: 0.01


# Evading

## Step 1: Benign Node Selection
### Identify candidate nodes with the same label as potentially malicious nodes.

In [13]:
def split_malicious_nodes_by_label(malicious_indices, labels):

    malicious_groups = defaultdict(list)

    for node_idx in malicious_indices:
        node_label = labels[node_idx]
        malicious_groups[node_label].append(node_idx)

    return dict(malicious_groups)

malicious_groups = split_malicious_nodes_by_label(indices_of_malicious_nodes, labels_test)

print("The number of malicious nodes for each label is as follows:")

for label, nodes in malicious_groups.items():
    print(f"Label {label}: {len(nodes)}")  


The number of malicious nodes for each label is as follows:
Label 8: 320
Label 0: 10


In [14]:
def split_nodes_by_label(phrases, labels):

    node_groups = defaultdict(list)

    for node_idx, node_features in enumerate(phrases):
        node_label = labels[node_idx]
        node_groups[node_label].append(node_idx)

    return dict(node_groups)

all_nodes_grouped = split_nodes_by_label(phrases_test, labels_test)

print("The number of nodes (either malicious or benign) for each label is as follows:")

for label, nodes in all_nodes_grouped.items():
    print(f"Label {label}: {len(nodes)}")  


The number of nodes (either malicious or benign) for each label is as follows:
Label 8: 7363
Label 9: 283478
Label 0: 169
Label 12: 112
Label 6: 2690
Label 11: 4060
Label 2: 427
Label 10: 34


In [15]:
def split_nodes_by_label_exclude_malicious(phrases, labels, indices_of_malicious_nodes):
    node_groups = defaultdict(list)
    malicious_set = set(indices_of_malicious_nodes)  

    for node_idx, node_features in enumerate(phrases):
        if node_idx not in malicious_set:
            node_label = labels[node_idx]
            node_groups[node_label].append(node_idx)

    return dict(node_groups)


all_nodes_grouped_excluding_malicious = split_nodes_by_label_exclude_malicious(phrases_test, labels_test, indices_of_malicious_nodes)

print("The number of candidate benign nodes for each label is as follows:")

for label, nodes in all_nodes_grouped_excluding_malicious.items():
    print(f"Label {label}: {len(nodes)}") 


The number of candidate benign nodes for each label is as follows:
Label 8: 7043
Label 9: 283478
Label 0: 159
Label 12: 112
Label 6: 2690
Label 11: 4060
Label 2: 427
Label 10: 34


## Step 2: Minimal Interaction Selection
### Prioritize nodes with the fewest interactions to reduce detectability.

In [ ]:
# def filter_nodes_by_phrase_length(node_groups, phrases, min_len=3, max_len=6): # 3-6
def filter_nodes_by_phrase_length(node_groups, phrases, min_len=1, max_len=3): # works here
# def filter_nodes_by_phrase_length(node_groups, phrases, min_len=20, max_len=40):

    filtered_groups = {}

    for label, node_indices in node_groups.items():
        # Apply filtering
        filtered = [idx for idx in node_indices if min_len <= len(phrases[idx]) <= max_len]
        
        # If filtering removes everything, keep original nodes
        if len(filtered) == 0:
            filtered_groups[label] = node_indices
        else:
            filtered_groups[label] = filtered

    return filtered_groups


filtered_nodes_grouped = filter_nodes_by_phrase_length(all_nodes_grouped_excluding_malicious, phrases_test)

print("The number of candidate benign nodes for each label is as follows:")

for label, nodes in filtered_nodes_grouped.items():
    print(f"Label {label}: {len(nodes)}")  


The number of candidate benign nodes for each label is as follows:
Label 8: 1400
Label 9: 30800
Label 0: 159
Label 12: 16
Label 6: 987
Label 11: 1152
Label 2: 246
Label 10: 25


## Step 3: Contextual Consistency Filtering
### Choose nodes with higher cosine similarity to malicious nodes to preserve the contextual  integrity and stealth.

In [29]:
def compute_top_similar_nodes(malicious_groups, filtered_nodes_grouped, nodes):
    top_nodes_by_similarity = {}

    for label in tqdm(malicious_groups.keys()):
        if label in filtered_nodes_grouped:
            malicious_indices = malicious_groups[label]
            filtered_indices = filtered_nodes_grouped[label]

            # Get feature vectors for malicious and filtered nodes
            malicious_vectors = np.array([nodes[idx] for idx in malicious_indices])
            filtered_vectors = np.array([nodes[idx] for idx in filtered_indices])

            # Compute cosine similarity (malicious nodes vs filtered nodes)
            similarities = cosine_similarity(filtered_vectors, malicious_vectors)  # Shape: (filtered, malicious)

            # Take the max similarity score for each filtered node
            max_similarities = similarities.max(axis=1)  # Max similarity per filtered node

            # Use np.argsort to get the indices that would sort the max_similarities array
            sorted_indices = np.array(filtered_indices)[np.argsort(-max_similarities)]  # Negative for descending order

            # Select top most similar nodes for the current label
            top_nodes_by_similarity[label] = sorted_indices

    return top_nodes_by_similarity

top_nodes_by_similarity = compute_top_similar_nodes(malicious_groups, filtered_nodes_grouped, nodes_test)

print("The number of candidate benign nodes for each label is as follows:")

for label, top_nodes in top_nodes_by_similarity.items():
    print(f"Label {label}: {len(top_nodes)}") 


100%|██████████| 2/2 [00:00<00:00, 593.72it/s]

The number of candidate benign nodes for each label is as follows:
Label 8: 1400
Label 0: 159


## Graph Structure Adjustment

In [30]:
# Prepare the DataFrame for modification
print("Number of rows in df before modification:", len(df_test))
filtered_nodes_grouped

malicious_to_benign = {}
for i in indices_of_malicious_nodes:

    if labels_test[i] in filtered_nodes_grouped and len(filtered_nodes_grouped[labels_test[i]]) > 0:
        malicious_to_benign[mapp_test[i]] = mapp_test[filtered_nodes_grouped[labels_test[i]][0]]


# Step 2: Retrieve all benign node IDs that will be replaced
all_benign_replacements = set(malicious_to_benign.values())

# Step 3: Extract relevant rows for each benign ID
benign_rows_dict = {benign_id: df_test[(df_test['actorID'] == benign_id) | (df_test['objectID'] == benign_id)].copy()
                    for benign_id in all_benign_replacements}

new_rows = []  # Store newly modified rows

for malicious_id, benign_id in tqdm(malicious_to_benign.items(), desc="Processing malicious nodes"):
    if benign_id in benign_rows_dict:
        modified_rows = benign_rows_dict[benign_id].copy()  # Copy rows where benign ID appears
        # If the benign node is in actorID, insert the malicious node in objectID
        modified_rows.loc[modified_rows['actorID'] == benign_id, 'actorID'] = malicious_id
        # If the benign node is in objectID, insert the malicious node in actorID
        modified_rows.loc[modified_rows['objectID'] == benign_id, 'objectID'] = malicious_id
        # Append modified rows to the new list
        new_rows.append(modified_rows)

# Step 5: Concatenate all new rows once for efficiency
df_curated = pd.concat([df_test] + new_rows, ignore_index=True)

print("Number of rows in df after modification:", len(df_curated))
print("The number of triggered events are: ", len(df_curated) - len(df_test))


Number of rows in df before modification: 798945


Processing malicious nodes: 100%|██████████| 330/330 [00:00<00:00, 2720.92it/s]


Number of rows in df after modification: 801265
The number of triggered events are:  2320


In [31]:
phrases_curated, labels_curated, edges_curated, mapp_curated = prepare_graph(df_curated)

nodes_curated = [infer(x) for x in phrases_curated]
nodes_curated = np.array(nodes_curated)
all_ids = list(df_curated['actorID']) + list(df_curated['objectID'])

In [ ]:
graph = Data(
    x=torch.tensor(nodes_curated, dtype=torch.float).to(device),
    y=torch.tensor(labels_curated, dtype=torch.long).to(device),
    edge_index=torch.tensor(edges_curated, dtype=torch.long).to(device)
)

graph.n_id = torch.arange(graph.num_nodes, device=device)

flag = torch.ones(graph.num_nodes, dtype=torch.bool, device=device)

for m_n in range(30,50):
    model.load_state_dict(torch.load(f'path to trained weights/lword2vec_gnn_five{m_n}_E3.pth',map_location=device))
    model.eval()
    out = model(graph.x, graph.edge_index)  # Directly use the full graph
    sorted, indices = out.sort(dim=1, descending=True)

    conf = (sorted[:, 0] - sorted[:, 1]) / sorted[:, 0]
    conf = (conf - conf.min()) / conf.max()  # normalize

    pred = indices[:, 0]
    cond = (pred == graph.y) & (conf > 0.8)
    flag[cond] = False

    correct = pred == graph.y  # boolean mask

indices_of_flagged_nodes = torch.nonzero(flag, as_tuple=False).view(-1).tolist()

print("number of nodes flagged as malicious: ", len(indices_of_flagged_nodes))

ids = set([mapp_curated[x] for x in indices_of_flagged_nodes]) 
TP, TN, FP, FN, prec, rec, fscore = helper(set(ids), set(all_ids), GT_mal, edges_curated, mapp_curated)


number of nodes flagged as malicious:  295363
True Positives: 207, True Negatives: 2829, False Positives: 287198, False Negatives: 123
Precision: 0.0, Recall: 0.63, Fscore: 0.0
False Positive Rate: 0.99, True Positive Rate: 0.63, True Negative Rate: 0.01


## Test Plausability

In [21]:
unique_exec_action_path_df = df_test[["exec", "action", "path"]].drop_duplicates()
print(len(unique_exec_action_path_df))

unique_exec_action_path_df = df_curated[["exec", "action", "path"]].drop_duplicates()
print(len(unique_exec_action_path_df))

12324
12324
